## **Exercise 04 : Enrichment and transformations**

### **04.1 Read the JSON file that you saved in ex02**

In [1]:
#  one of the columns has the float type, so let us define the format of it in pandas using pd.options.display.float_format: floats with two decimals
import pandas as pd
import numpy as np
pd.options.display.float_format = '{:.2f}'.format
df = pd.read_json('../../datasets/auto.json', orient="records")
df.set_index('CarNumber', inplace=True)
df['Refund'] = df['Refund'].astype(float)
# there are values missing from the Model, do not do anything with them
df

,Refund,Fines,Make,Model
CarNumber,,,,
Y163O8161RUS,2.00,3200.00,Ford,Focus
E432XX77RUS,1.00,6500.00,Toyota,Camry
7184TT36RUS,1.00,2100.00,Ford,Focus
X582HE161RUS,2.00,2000.00,Ford,Focus
92918M178RUS,1.00,5700.00,Ford,Focus
...,...,...,...,...
Y163O8161RUS,2.00,1600.00,Ford,Focus
M0309X197RUS,1.00,22300.00,Ford,Focus
O673E8197RUS,2.00,600.00,Ford,Focus


### **04.2 Enrich the dataframe using a sample from that dataframe**

In [2]:
# create a sample with 200 new observations with random_state = 21
#    *the sample should not have new combinations of the car number, make and model, so the whole dataset will be consistent in these terms
my_sample = df.sample(n=200, random_state=42, replace=True)
#    *there are no restrictions on the refund and fines, you can take any value from these columns at random and use it towards any car number
np.random.seed(42)
refund_set = set(df['Refund'].to_list())
fines_set = set(df['Fines'].to_list())
my_sample['Refund'] = np.random.choice(list(refund_set), size=200).astype(float)
my_sample['Fines'] = np.random.choice(list(fines_set), size=200).astype(float)
my_sample

,Refund,Fines,Make,Model
CarNumber,,,,
O83797197RUS,1.00,40600.00,Ford,Focus
E396MO152RUS,2.00,6900.00,Ford,Focus
O752MX161RUS,1.00,3200.00,Ford,Focus
9583EY178RUS,1.00,6900.00,Ford,Focus
7788KT197RUS,1.00,7000.00,Ford,Focus
...,...,...,...,...
7831C8197RUS,2.00,2200.00,Ford,Focus
E53677152RUS,2.00,7800.00,Ford,Focus
H908YK197RUS,2.00,22400.00,Ford,Focus


In [3]:
#  concatenate the sample with the initial dataframe to a new dataframe concat_rows
concat_rows = pd.concat([df, my_sample])
concat_rows

,Refund,Fines,Make,Model
CarNumber,,,,
Y163O8161RUS,2.00,3200.00,Ford,Focus
E432XX77RUS,1.00,6500.00,Toyota,Camry
7184TT36RUS,1.00,2100.00,Ford,Focus
X582HE161RUS,2.00,2000.00,Ford,Focus
92918M178RUS,1.00,5700.00,Ford,Focus
...,...,...,...,...
7831C8197RUS,2.00,2200.00,Ford,Focus
E53677152RUS,2.00,7800.00,Ford,Focus
H908YK197RUS,2.00,22400.00,Ford,Focus


### **04.3 Enrich the dataframe concat_rows by a new column with the data generated**

In [4]:
# create a series with the name Year using random integers from 1980 to 2019, use np.random.seed(21) before generating the years
np.random.seed(42)
year = pd.Series(np.random.choice(range(1980, 2020), size=concat_rows.shape[0]))
year

0      2018
1      2008
2      1994
3      1987
4      2000
       ... 
920    2004
921    1996
922    2012
923    1981
924    1993
Length: 925, dtype: int64

In [5]:
# concatenate the series with the dataframe and name it fines
fines = pd.concat([concat_rows.reset_index(), pd.DataFrame(year, columns=['Year'])], axis=1) 
fines.set_index('CarNumber', inplace=True)

### **04.4 Enrich the dataframe with the data from another dataframe**

In [6]:
# create a new dataframe with the car numbers and their owners
#    *get the most popular surnames (you can find the file surname.json in the attachments) in the US
df2 = pd.read_json('../../datasets/surname.json', orient="records", )
headers = list(df2.iloc[0])
df2.drop(0, inplace=True)
df2.columns = headers
df2['COUNT'] = df2['COUNT'].astype(int)
# to improve random with prob of each name with count weights
df2['p'] = df2['COUNT'] / df2['COUNT'].sum()

#    *create a new series with the surnames (they should not have special characters like commas, brackets, etc.) from the data you gathered, 
#        the count should be equal to the number of unique car numbers using the sample (use random_state = 21)
np.random.seed(42)
df2['NAME'] = df2['NAME'].str.replace('\W', '')
surnames = pd.Series(np.random.choice(df2['NAME'], 
                                      size=len(fines.index.drop_duplicates()),
                                      p=df2['p']))
 
#    *create the dataframe owners with 2 columns: CarNumber and SURNAME
owners = pd.concat([pd.DataFrame(fines.index.drop_duplicates()), 
                    pd.DataFrame(surnames, columns=['Name'])], axis=1) 
owners.set_index('CarNumber', inplace=True)
owners

,Name
CarNumber,
Y163O8161RUS,JOHNSON
E432XX77RUS,WILLIAMS
7184TT36RUS,RODRIGUEZ
X582HE161RUS,MYERS
92918M178RUS,DAVIS
...,...
O136HO197RUS,JONES
O22097197RUS,GUTIERREZ
M0309X197RUS,BROOKS


In [7]:
# append 5 more observations to the fines dataframe (come up with your own ideas of CarNumber, etc.)
new_rows = pd.DataFrame({"Refund": [1,2,1,2,1], 
                         "Fines": [100_000, 1_000, 22_000, 4_000, 10_000],
                         "Make": ['KAMAZ', 'VAZ', 'KAMAZ', 'VAZ', 'KAMAZ'],
                         "Model": ['49252', 'Baklajan', '1984', 'Lastochka', '49252'],
                         "Year": [2019, 2019, 2019,2019, 2019]}, index=['a','b', 'c', 'd', 'e'])
fines = pd.concat([fines, new_rows])
fines

,Refund,Fines,Make,Model,Year
Y163O8161RUS,2.00,3200.00,Ford,Focus,2018
E432XX77RUS,1.00,6500.00,Toyota,Camry,2008
7184TT36RUS,1.00,2100.00,Ford,Focus,1994
X582HE161RUS,2.00,2000.00,Ford,Focus,1987
92918M178RUS,1.00,5700.00,Ford,Focus,2000
...,...,...,...,...,...
a,1.00,100000.00,KAMAZ,49252,2019
b,2.00,1000.00,VAZ,Baklajan,2019
c,1.00,22000.00,KAMAZ,1984,2019
d,2.00,4000.00,VAZ,Lastochka,2019


In [8]:
# delete the dataframe last 20 observations from the owners and add 3 new observations (they are not the same as those you add to the fines dataframe
owners.drop(owners.tail(20).index,inplace=True)
new_rows = pd.DataFrame({"Name": ['Ya-Ya','Coco','Jambo']}, 
                         index=['f','g', 'h'])
owners = pd.concat([owners, new_rows])
owners

,Name
Y163O8161RUS,JOHNSON
E432XX77RUS,WILLIAMS
7184TT36RUS,RODRIGUEZ
X582HE161RUS,MYERS
92918M178RUS,DAVIS
...,...
O50197197RUS,RODRIGUEZ
7608EE777RUS,NGUYEN
f,Ya-Ya
g,Coco


In [9]:
# join both dataframes:
#    *the new dataframe should have only the car numbers that exist in both dataframes
fines.join(owners, how='inner')

,Refund,Fines,Make,Model,Year,Name
Y163O8161RUS,2.00,3200.00,Ford,Focus,2018,JOHNSON
E432XX77RUS,1.00,6500.00,Toyota,Camry,2008,WILLIAMS
7184TT36RUS,1.00,2100.00,Ford,Focus,1994,RODRIGUEZ
X582HE161RUS,2.00,2000.00,Ford,Focus,1987,MYERS
92918M178RUS,1.00,5700.00,Ford,Focus,2000,DAVIS
...,...,...,...,...,...,...
7831C8197RUS,2.00,2200.00,Ford,Focus,2004,PETERSON
E53677152RUS,2.00,7800.00,Ford,Focus,1996,RIVERA
H908YK197RUS,2.00,22400.00,Ford,Focus,2012,JOHNSON
T6439O50RUS,1.00,8200.00,Ford,Focus,1981,RODRIGUEZ


In [10]:
#    *the new dataframe should have all the car numbers that exist in both dataframes
fines.join(owners, how='outer')

,Refund,Fines,Make,Model,Year,Name
704687163RUS,2.00,1400.00,Ford,Focus,1986.00,CAMPBELL
704787163RUS,2.00,2800.00,Ford,Focus,2010.00,MOORE
704987163RUS,2.00,8594.60,Ford,Focus,2018.00,NGUYEN
705287163RUS,2.00,2000.00,Ford,Focus,1990.00,GARCIA
705387163RUS,2.00,700.00,Ford,Focus,1984.00,WILSON
...,...,...,...,...,...,...
d,2.00,4000.00,VAZ,Lastochka,2019.00,NaN
e,1.00,10000.00,KAMAZ,49252,2019.00,NaN
f,NaN,NaN,NaN,NaN,NaN,Ya-Ya
g,NaN,NaN,NaN,NaN,NaN,Coco


In [11]:
#    *the new dataframe should have only the car numbers from the fines dataframe
fines.join(owners, how='left')

,Refund,Fines,Make,Model,Year,Name
Y163O8161RUS,2.00,3200.00,Ford,Focus,2018,JOHNSON
E432XX77RUS,1.00,6500.00,Toyota,Camry,2008,WILLIAMS
7184TT36RUS,1.00,2100.00,Ford,Focus,1994,RODRIGUEZ
X582HE161RUS,2.00,2000.00,Ford,Focus,1987,MYERS
92918M178RUS,1.00,5700.00,Ford,Focus,2000,DAVIS
...,...,...,...,...,...,...
a,1.00,100000.00,KAMAZ,49252,2019,NaN
b,2.00,1000.00,VAZ,Baklajan,2019,NaN
c,1.00,22000.00,KAMAZ,1984,2019,NaN
d,2.00,4000.00,VAZ,Lastochka,2019,NaN


In [12]:
#    *the new dataframe should have only the car numbers from the owners dataframe
fines.join(owners, how='right')

,Refund,Fines,Make,Model,Year,Name
Y163O8161RUS,2.00,3200.00,Ford,Focus,2018.00,JOHNSON
Y163O8161RUS,2.00,1600.00,Ford,Focus,1995.00,JOHNSON
E432XX77RUS,1.00,6500.00,Toyota,Camry,2008.00,WILLIAMS
E432XX77RUS,2.00,13000.00,Toyota,Camry,1999.00,WILLIAMS
E432XX77RUS,2.00,29700.00,Toyota,Camry,1997.00,WILLIAMS
...,...,...,...,...,...,...
7608EE777RUS,1.00,4000.00,Skoda,Octavia,2017.00,NGUYEN
7608EE777RUS,1.00,4100.00,Skoda,Octavia,1993.00,NGUYEN
f,NaN,NaN,NaN,NaN,NaN,Ya-Ya
g,NaN,NaN,NaN,NaN,NaN,Coco


### **04.5 Create a pivot table from the fines dataframe, it should look like this (the values are the sums of the fines), but with all the years**

In [13]:
fines.pivot_table(index=['Make', 'Model'], columns=["Year"], values="Fines", aggfunc="sum")

Year                      1980      1981      1982      1983      1984  \
Make       Model                                                         
Ford       Focus     141500.00 191700.00 281094.60 411994.60  93394.60   
           Mondeo          NaN       NaN       NaN       NaN   1100.00   
KAMAZ      1984            NaN       NaN       NaN       NaN       NaN   
           49252           NaN       NaN       NaN       NaN       NaN   
Skoda      Octavia     1800.00  17189.20   5500.00   2000.00  22000.00   
Toyota     Camry           NaN       NaN       NaN    800.00   2000.00   
           Corolla    17200.00       NaN       NaN  12994.60       NaN   
VAZ        Baklajan        NaN       NaN       NaN       NaN       NaN   
           Lastochka       NaN       NaN       NaN       NaN       NaN   
Volkswagen Golf            NaN   1000.00   5000.00  20800.00 168000.00   
           Jetta           NaN       NaN       NaN       NaN   7500.00   
           Passat      8594.60  18500.00       NaN       NaN   9900.00   
           Touareg         NaN  22400.00       NaN   5800.00       NaN   

Year                      1985     1986      1987      1988      1989  ...  \
Make       Model                                                       ...   
Ford       Focus     103700.00 71100.00 350583.80 148094.60 143300.00  ...   
           Mondeo          NaN      NaN       NaN       NaN       NaN  ...   
KAMAZ      1984            NaN      NaN       NaN       NaN       NaN  ...   
           49252           NaN      NaN       NaN       NaN       NaN  ...   
Skoda      Octavia         NaN      NaN   2600.00   3000.00       NaN  ...   
Toyota     Camry           NaN 12000.00       NaN       NaN       NaN  ...   
           Corolla    50400.00  3200.00       NaN   4000.00  27800.00  ...   
VAZ        Baklajan        NaN      NaN       NaN       NaN       NaN  ...   
           Lastochka       NaN      NaN       NaN       NaN       NaN  ...   
Volkswagen Golf            NaN   200.00       NaN       NaN       NaN  ...   
           Jetta           NaN      NaN       NaN       NaN  46000.00  ...   
           Passat          NaN      NaN   1600.00       NaN   8594.60  ...   
           Touareg     8594.60      NaN       NaN       NaN       NaN  ...   

Year                     2010      2011      2012      2013      2014  \
Make       Model                                                        
Ford       Focus     92700.00 109594.60 480894.60 119700.00 125494.60   
           Mondeo         NaN       NaN       NaN       NaN       NaN   
KAMAZ      1984           NaN       NaN       NaN       NaN       NaN   
           49252          NaN       NaN       NaN       NaN       NaN   
Skoda      Octavia    3500.00  54594.60   9894.60   3900.00       NaN   
Toyota     Camry     18300.00       NaN       NaN   3500.00  12994.60   
           Corolla        NaN       NaN  16000.00  12700.00   7800.00   
VAZ        Baklajan       NaN       NaN       NaN       NaN       NaN   
           Lastochka      NaN       NaN       NaN       NaN       NaN   
Volkswagen Golf       9300.00       NaN   5800.00    300.00  26000.00   
           Jetta          NaN       NaN       NaN       NaN   4000.00   
           Passat    16500.00   3000.00    600.00  16600.00 148400.00   
           Touareg        NaN       NaN  11000.00       NaN   6300.00   

Year                      2015      2016      2017      2018      2019  
Make       Model                                                        
Ford       Focus     106100.00 198894.60 174089.20 210478.40 203800.00  
           Mondeo          NaN  34400.00  29700.00       NaN       NaN  
KAMAZ      1984            NaN       NaN       NaN       NaN  22000.00  
           49252           NaN       NaN       NaN       NaN 110000.00  
Skoda      Octavia     2000.00  33594.60   5800.00       NaN  11594.60  
Toyota     Camry       1000.00    500.00       NaN       NaN       NaN  
           Corolla         NaN   8300.

### **04.6 save both the fines and owners dataframes to CSV files without an index**

In [14]:
fines.to_csv("../../datasets/fines.csv", sep=';', index_label='CarNumber')
owners.to_csv("../../datasets/owners.csv", sep=';', index_label='CarNumber')